In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import joblib
from rdkit import Chem
from rdkit.Chem import AllChem  
import os
from torch_geometric.data import Data, DataLoader
import pandas as pd
import numpy as np

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    def __init__(self, input_size, hidden_sizes, output_size, dropout_rate=0.3):
        
        super(MLP, self).__init__()
        layers = []
        in_dim = input_size  
        
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(in_dim, hidden_size))            
            layers.append(nn.BatchNorm1d(hidden_size))                 
            layers.append(nn.ReLU())                                   
            layers.append(nn.Dropout(dropout_rate))                    
            in_dim = hidden_size                                      
        
        
        layers.append(nn.Linear(in_dim, output_size))
        
        
        self.network = nn.Sequential(*layers)

    def forward(self, data):
        
        x = data.morgan_fp  
        out = self.network(x)  
        return out

In [ ]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
input_size = 2048
hidden_sizes = [256, 64]
output_size = 1
model_conductivity_MLP = MLP(input_size=input_size, hidden_sizes=hidden_sizes, output_size=output_size).to(device) 

In [ ]:

IL_ECW_xgb_model_path = '/path/to/example/ionic_liquid/static/model/IL_ECW_xgb_model_fp.joblib'
IL_ECW_xgb_model = joblib.load(IL_ECW_xgb_model_path)
print("Public message removed for release.")


Tm_xgb_model_path = '/path/to/example/ionic_liquid/static/model/Tm_xgb_model_fp.joblib'
Tm_xgb_model = joblib.load(Tm_xgb_model_path)
print("Public message removed for release.")


MLP_model_path = '/path/to/example/ionic_liquid/static/model/conductivity_MLP_model_fp.pt'

state_dict = torch.load(MLP_model_path, map_location=device)
model_conductivity_MLP.load_state_dict(state_dict)
print("Public message removed for release.")

In [ ]:

def add_hydrogens_to_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)  
    mol = Chem.AddHs(mol)  
    smiles_with_h = Chem.MolToSmiles(mol)  
    return smiles_with_h

In [ ]:
def smiles_to_morgan_fingerprint(smiles, radius=2, n_bits=2048):
    
    
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Public message removed for release.")
    
    
    fingerprint = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    
    
    array = list(fingerprint.ToBitString())  
    array = [int(bit) for bit in array]  
    
    
    tensor = torch.tensor(array, dtype=torch.float32).unsqueeze(0)  
    
    return tensor

In [ ]:
def create_morgan_fp_tensor(smiles):
    
    smiles = add_hydrogens_to_smiles(smiles) 
    
    
    morgan_fp_tensor = smiles_to_morgan_fingerprint(smiles, radius=2, n_bits=2048)
        
    return morgan_fp_tensor

In [ ]:
smiles = "C[N+](C)(C)CCO.CCCCCC(=O)[O-]"
test_morgan_fp_tensor = create_morgan_fp_tensor(smiles)


data = Data(
    smiles=smiles,
    morgan_fp = test_morgan_fp_tensor,
)


y_ECW = IL_ECW_xgb_model.predict(test_morgan_fp_tensor)
y_Tm = Tm_xgb_model.predict(test_morgan_fp_tensor)


model_conductivity_MLP.eval()
y_conductivity = []

data = data.to(device)
data.morgan_fp = data.morgan_fp.float()  
out = model_conductivity_MLP(data)

y_conductivity.append(out.detach().cpu().numpy())
y_conductivity = np.concatenate(y_conductivity, axis=0)
y_conductivity = y_conductivity.flatten()  

In [ ]:
print(y_ECW)
print(y_Tm)
print(y_conductivity)